Env setup and import

In [ ]:
import torch
from torch import nn
from torchrl.collectors import MultiSyncCollector
from torchrl.data.replay_buffers import ReplayBuffer
from torchrl.data.replay_buffers.samplers import SamplerWithoutReplacement
from torchrl.data.replay_buffers.storages import LazyTensorStorage
from torchrl.objectives import ClipPPOLoss, ValueEstimators
from env_simplified import make_env
from ai_setup import make_policy_critic

# 1. 宣告這是一個返回 TorchRL 原生環境（EnvBase）的函式
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
env = make_env()
policy, critic = make_policy_critic(env, 'policy_net_third.pth', 'critic_net_third.pth')

cuda


c:\Users\ctc73\PycharmProjects\MahjongAI\.venv\Lib\site-packages\torchrl\envs\libs\pettingzoo.py:281: UserWarning: PettingZoo in TorchRL is tested using version == 1.24.3 , If you are using a different version and are experiencing compatibility issues,please raise an issue in the TorchRL github.
  warnings.warn(


Rollout

In [ ]:
# import pygame
# from pygame_visualizer import render_game_state

# pygame.init()
# data = env.rollout(200, policy=policy.to('cpu'))
# screen = pygame.display.set_mode(size=(800, 800))
# font = pygame.font.Font("C:/Windows/Fonts/seguisym.ttf", 48)
# from sys import exit
# while True:
#     for event in pygame.event.get():
#         if event.type == pygame.QUIT:
#             pygame.quit()
#             exit()
#     screen.fill('white')
#     render_game_state(env._env.gamestate, screen, font)
#     pygame.display.update()

Loss function and optimizer

In [9]:
policy = policy.to(device)
loss_module = ClipPPOLoss(
    actor_network=policy, # type: ignore
    critic_network=critic,
    entropy_coeff=0.01
)
loss_module.set_keys(  # We have to tell the loss where to find the keys
    reward=env.reward_key,
    action=env.action_key,
    value=("agents", "state_value"),
    # These last 2 keys will be expanded to match the reward shape
    done=("agents", "done"),                # per-agent
    terminated=("agents", "terminated"),
)
gamma = 0.995  # discount factor
lmbda = 0.9  # lambda for generalised advantage estimation
lr = 5e-5
loss_module.make_value_estimator(
    ValueEstimators.GAE, gamma=gamma, lmbda=lmbda
)  
GAE = loss_module.value_estimator

optim = torch.optim.Adam(loss_module.parameters(), lr)

loss_module = loss_module.to(device)

Train loop

In [12]:
num_epochs = 5
max_grad_norm = 0.1
frames_per_batch = 1600  # Number of team frames collected per training iteration
n_iters = 3 # Number of sampling and training iterations
total_frames = frames_per_batch * n_iters
minibatch_size = 800

if __name__ == "__main__":
    replay_buffer = ReplayBuffer(
        storage=LazyTensorStorage(
            frames_per_batch, device=device
        ),  # We store the frames_per_batch collected at each iteration
        sampler=SamplerWithoutReplacement(),
        batch_size=minibatch_size,  # We will sample minibatches of this siz
    )
    policy=policy.to(device)
    collector = MultiSyncCollector(
        [make_env] * 12,
        policy=policy,
        device='cpu',
        storing_device=device,
        frames_per_batch=frames_per_batch,
        total_frames=total_frames,
        cat_results=0
    )
    from tqdm.auto import tqdm
    for it, tensordict_data in enumerate(tqdm(collector)):
        tensordict_data.set(
            ("next", "agents", "done"),
            tensordict_data.get(("next", "done"))
            .unsqueeze(-1)
            .expand(tensordict_data.get_item_shape(("next", env.reward_key))),
        )
        tensordict_data.set(
            ("next", "agents", "terminated"),
            tensordict_data.get(("next", "terminated"))
            .unsqueeze(-1)
            .expand(tensordict_data.get_item_shape(("next", env.reward_key))),
        )
        # We need to expand the done and terminated to match the reward shape (this is expected by the value estimator)

        with torch.no_grad():
            GAE(
                tensordict_data,
                params=loss_module.critic_network_params,
                target_params=loss_module.target_critic_network_params,
            )  # Compute GAE and add it to the data

        data_view = tensordict_data.reshape(-1)  # Flatten the batch size to shuffle data
        replay_buffer.extend(data_view)

        for _ in range(num_epochs):
            for _ in range(frames_per_batch // minibatch_size):
                subdata = replay_buffer.sample()
                subdata = subdata.to(device)
                loss_vals = loss_module(subdata)

                loss_value = (
                    loss_vals["loss_objective"]
                    + loss_vals["loss_critic"]
                    + loss_vals["loss_entropy"]
                )

                loss_value.backward()

                torch.nn.utils.clip_grad_norm_(
                    loss_module.parameters(), max_grad_norm
                )  # Optional

                optim.step()
                optim.zero_grad()

        collector.update_policy_weights_()


c:\Users\ctc73\PycharmProjects\MahjongAI\.venv\Lib\site-packages\torchrl\collectors\_multi_sync.py:202: UserWarning: frames_per_batch 1600 is not exactly divisible by the number of collector workers 12, this results in more frames_per_batch per iteration that requested.To silence this message, set the environment variable RL_WARNINGS to False.
  warnings.warn(
  0%|          | 0/3 [00:00<?, ?it/s]

2026-07-26 10:58:38,910 [torchrl][WARNING]    use_buffer not specified and not yet inferred from data, assuming `True`. [END]
2026-07-26 10:58:39,762 [torchrl][INFO]    Initialized LazyTensorStorage with torch.Size([1600]) shape [END]


 33%|███▎      | 1/3 [03:37<07:14, 217.32s/it]


KeyboardInterrupt: 

Rollout per step

In [20]:
import pygame
from sys import exit
from pygame_visualizer import render_game_state
from torchrl.envs.utils import step_mdp  # <-- Added missing utility

pygame.init()
screen = pygame.display.set_mode(size=(800, 800))
font = pygame.font.Font("C:/Windows/Fonts/seguisym.ttf", 48)

td = env.reset()
policy = policy.to('cpu')

while True:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            pygame.quit()
            exit()
            
        # Step through the game manually by pressing SPACE
        if event.type == pygame.KEYDOWN and event.key == pygame.K_SPACE:
            # 1. Check if the game is already over before stepping
            if td.get("done", torch.tensor([False])).any():
                print("Episode finished! Resetting environment...")
                td = env.reset()
                continue
                
            # 2. Policy reads root "observation" and writes root "action" into td
            td = policy(td)
            
            # 3. Environment executes the action and creates the "next" sub-TensorDict
            td = env.step(td)
            
            # 4. Check if this new step ended the game
            if td[("next", "done")].any():
                print(f"Game Over! Reward: {td[('next', 'agents', 'reward')]}")
            
            # 5. Move "next" keys to root level so the policy can read them next turn
            td = step_mdp(td)
            
    screen.fill('white')
    # Render the internal unwrapped environment game state
    render_game_state(env._env.gamestate, screen, font)
    pygame.display.update()

SystemExit: 